# RNNG Training & Eval — Colab (T4 GPU)

**Before running:** `Runtime > Change runtime type > T4 GPU`.

### What this notebook does
1. Clones the project repo and `aistairc/rnng-pytorch`
2. Parses the clean corpus **once** with benepar → PTB trees (saved to Drive for resumability)
3. Swaps verb terminals per condition to produce 5 × noised tree sets
4. Runs `preprocess.py` → JSON training files
5. Trains RNNG for each condition × seed on the T4
6. Evaluates via `beam_search.py` on the held-out minimal-pair test set
7. Writes `results/rnng_eval_results.csv` (same schema as the LSTM CSV, plus a `model` column)

All expensive intermediates (parsed trees, checkpoints) are written to **Drive** so the
notebook survives session disconnects.  Each step skips if its output already exists.

### After running
Commit `results/rnng_eval_results.csv` and push to the repo.

In [ ]:
# 0 — GPU check
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## Step 1 — Config
Edit `DATA_DIR` and `DRIVE_DIR` to match your Drive layout, then run all cells top to bottom.

In [ ]:
# 1 — Config (edit the two paths below)
DATA_DIR    = '/content/drive/MyDrive/dsc291_data'    # folder containing the 6 TSVs
DRIVE_DIR   = '/content/drive/MyDrive/dsc291_rnng'    # folder for all RNNG intermediates

REPO_URL    = 'https://github.com/blue-octopus235/FinalProject-LMs-CogModels-DSC291.git'
REPO_DIR    = '/content/FinalProject-LMs-CogModels-DSC291'
RNNG_DIR    = '/content/rnng-pytorch'

MAX_SENTENCES = 150_000   # keep constant across all conditions
EPOCHS        = 15        # reduce to 10 if T4 time is tight
SEEDS         = [1, 2, 3] # reduce to [1] for a quick smoke test
CONDITIONS    = ['baseline', 'low', 'medium_low', 'medium_high', 'high']

PARSE_BATCH   = 64        # sentences per benepar batch

# beam_search.py eval settings (reduce beam/batch if OOM)
EVAL_BEAM       = 100
EVAL_WORD_BEAM  = 20
EVAL_SHIFT      = 5
EVAL_BATCH      = 50
EVAL_STACK      = 50

CONDITIONS_FILES = {
    'baseline':    'sva_corpus_baseline_rate_0.0.tsv',
    'low':         'sva_corpus_low_rate_0.004.tsv',
    'medium_low':  'sva_corpus_medium_low_rate_0.02.tsv',
    'medium_high': 'sva_corpus_medium_high_rate_0.1.tsv',
    'high':        'sva_corpus_high_rate_0.5.tsv',
}
RATES = {'baseline': 0.0, 'low': 0.004, 'medium_low': 0.02, 'medium_high': 0.10, 'high': 0.50}
print('Config OK')

## Step 2 — Mount Drive

In [ ]:
# 2 — Mount Drive + create dirs
from google.colab import drive
drive.mount('/content/drive')

import os
for d in [DRIVE_DIR,
          os.path.join(DRIVE_DIR, 'parsed_trees'),
          os.path.join(DRIVE_DIR, 'checkpoints'),
          os.path.join(DRIVE_DIR, 'results'),
          os.path.join(DRIVE_DIR, 'eval_tmp')]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted. Data dir contents:')
print(sorted(os.listdir(DATA_DIR)))

## Step 3 — Clone repo + install dependencies

In [ ]:
# 3a — Clone project repo
import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists, pulling latest...')
    !cd {REPO_DIR} && git pull
%cd {REPO_DIR}

In [ ]:
# 3b — Install Python dependencies
!pip install -q lemminflect benepar "spacy>=3,<4" nltk tensorboard tqdm
!python -m spacy download en_core_web_md -q
import benepar
benepar.download('benepar_en3')
print('Dependencies installed.')

## Step 4 — Clone + patch aistairc/rnng-pytorch

In [ ]:
# 4 — Clone rnng-pytorch and apply the two bug-fix patches
if not os.path.exists(RNNG_DIR):
    !git clone https://github.com/aistairc/rnng-pytorch.git {RNNG_DIR}
    !cd {RNNG_DIR} && git apply {REPO_DIR}/rnng/rnng_pytorch_fixes.patch
    print('Cloned and patched.')
else:
    print(f'{RNNG_DIR} already exists, assuming already patched.')

## Step 5 — Parse clean corpus with benepar (one-time, ~25–45 min on T4)

We parse the **baseline** (clean) training sentences once and save the PTB trees to Drive.
On subsequent runs this cell loads the cached trees instantly.

The parse captures sentence structure from the clean verbs.  Noised conditions just get the
verb terminal swapped in Step 6 — the bracketing is identical across all 5 conditions.

In [ ]:
# 5 — Parse clean corpus
import sys, csv, json, re
sys.path.insert(0, f'{REPO_DIR}/src')
from data_utils import get_test_indices

PARSE_DIR        = os.path.join(DRIVE_DIR, 'parsed_trees')
CLEAN_TREES_FILE = os.path.join(PARSE_DIR, 'clean_trees.txt')
PARSE_META_FILE  = os.path.join(PARSE_DIR, 'parse_meta.json')

csv.field_size_limit(2**31 - 1)

if os.path.exists(PARSE_META_FILE):
    print('Loading cached parse results from Drive...')
    with open(PARSE_META_FILE) as f:
        parse_meta = json.load(f)
    with open(CLEAN_TREES_FILE) as f:
        clean_trees = [line.rstrip('\n') for line in f]
    print(f'Loaded {len(clean_trees)} trees.')
else:
    import spacy, benepar
    from tqdm.auto import tqdm

    test_idx = get_test_indices()
    baseline_path = os.path.join(DATA_DIR, 'sva_corpus_baseline_rate_0.0.tsv')

    # Collect training rows from the baseline (clean) corpus
    train_rows = []
    with open(baseline_path) as f:
        reader = csv.DictReader(f, delimiter='\t')
        for i, row in enumerate(reader):
            if i in test_idx:
                continue
            s = row.get('orig_sentence', '').strip()
            if not s:
                continue
            try:
                vidx = int(row['verb_index']) - 1  # 0-based
            except (KeyError, ValueError):
                continue
            toks = s.split()
            if not (0 <= vidx < len(toks)):
                continue
            train_rows.append({
                'row_idx':   i,
                'sent':      s,
                'verb_idx':  vidx,
                'verb_form': toks[vidx],
                'verb_pos':  row.get('verb_pos', ''),
            })
            if len(train_rows) >= MAX_SENTENCES:
                break
    print(f'Collected {len(train_rows)} training sentences from baseline corpus.')

    # Parse with benepar
    nlp = spacy.load('en_core_web_md')
    nlp.add_pipe('benepar', config={'model': 'benepar_en3'})

    parse_meta = []  # [{row_idx, verb_idx, verb_form, verb_pos, tree_line}, ...]
    clean_trees = []
    fail_count = 0

    for start in tqdm(range(0, len(train_rows), PARSE_BATCH), desc='Parsing'):
        batch = train_rows[start:start + PARSE_BATCH]
        try:
            docs = list(nlp.pipe([r['sent'] for r in batch]))
        except Exception:
            docs = [None] * len(batch)
        for row, doc in zip(batch, docs):
            if doc is None:
                fail_count += 1
                continue
            try:
                sent_obj = list(doc.sents)[0]
                ps = sent_obj._.parse_string
                if ps.startswith('('):
                    parse_meta.append({
                        'row_idx':   row['row_idx'],
                        'verb_idx':  row['verb_idx'],
                        'verb_form': row['verb_form'],
                        'verb_pos':  row['verb_pos'],
                        'tree_line': len(clean_trees),
                    })
                    clean_trees.append(ps)
                else:
                    fail_count += 1
            except Exception:
                fail_count += 1

    print(f'Parsed {len(clean_trees)}/{len(train_rows)}, {fail_count} failures.')

    # Save to Drive
    with open(CLEAN_TREES_FILE, 'w') as f:
        for tree in clean_trees:
            f.write(tree.strip() + '\n')
    with open(PARSE_META_FILE, 'w') as f:
        json.dump(parse_meta, f)
    print(f'Saved to {PARSE_DIR}.')

## Step 6 — Per-condition: swap verb terminals → preprocess.py

For each noised condition, we read the noised corpus at the same training rows,
swap the verb terminal in each benepar tree where the verb changed, and run
`preprocess.py` to produce the JSON training files RNNG expects.

The tree *structure* (all non-terminals and brackets) is identical across all 5 conditions;
only the single verb terminal changes, so we only parsed once in Step 5.

In [ ]:
# 6 — Verb swap + preprocess.py for each condition
import subprocess, re

def swap_verb_in_tree(tree_str, verb_idx_0based, noised_verb):
    """Replace the verb terminal at 0-based word position, flipping VBP<->VBZ."""
    terminals = list(re.finditer(r'\((\S+)\s+([^\s()]+)\)', tree_str))
    if verb_idx_0based >= len(terminals):
        return tree_str  # safety: index out of range, keep original
    m = terminals[verb_idx_0based]
    old_tag = m.group(1)
    new_tag = 'VBZ' if old_tag == 'VBP' else ('VBP' if old_tag == 'VBZ' else old_tag)
    return tree_str[:m.start()] + f'({new_tag} {noised_verb})' + tree_str[m.end():]

# Build lookup: row_idx -> parse_meta entry
row_to_meta       = {m['row_idx']: m for m in parse_meta}
selected_row_idxs = set(row_to_meta.keys())

for cond in CONDITIONS:
    cond_dir    = os.path.join(PARSE_DIR, cond)
    json_train  = os.path.join(cond_dir, f'{cond}-train.json')

    if os.path.exists(json_train):
        print(f'[{cond}] preprocessed JSON already exists — skipping.')
        continue

    os.makedirs(cond_dir, exist_ok=True)
    print(f'[{cond}] Reading noised corpus and building condition-specific trees...')

    # Read noised verb forms for all selected rows
    cond_path = os.path.join(DATA_DIR, CONDITIONS_FILES[cond])
    noised_verb_map = {}  # row_idx -> noised verb surface form
    with open(cond_path) as f:
        reader = csv.DictReader(f, delimiter='\t')
        for i, row in enumerate(reader):
            if i in selected_row_idxs:
                toks = row.get('orig_sentence', '').split()
                vidx = row_to_meta[i]['verb_idx']
                if 0 <= vidx < len(toks):
                    noised_verb_map[i] = toks[vidx]

    # Build condition-specific tree list (in parse_meta order)
    cond_trees = []
    swap_count = 0
    for meta in parse_meta:
        clean_tree  = clean_trees[meta['tree_line']]
        noised_verb = noised_verb_map.get(meta['row_idx'], meta['verb_form'])
        if noised_verb != meta['verb_form']:
            cond_trees.append(swap_verb_in_tree(clean_tree, meta['verb_idx'], noised_verb))
            swap_count += 1
        else:
            cond_trees.append(clean_tree)
    print(f'  {swap_count}/{len(cond_trees)} verb terminals swapped.')

    # Train / val / dummy-test split  (90 / 9 / 1)
    n_total = len(cond_trees)
    n_val   = max(200, int(n_total * 0.1))
    n_test  = min(200, n_val)        # dummy test file — only used by preprocess.py for NTs
    train_trees = cond_trees[n_val:]
    val_trees   = cond_trees[:n_val]
    test_trees  = val_trees[:n_test]

    train_file = os.path.join(cond_dir, 'train.txt')
    val_file   = os.path.join(cond_dir, 'val.txt')
    test_file  = os.path.join(cond_dir, 'test.txt')
    for path, trees in [(train_file, train_trees), (val_file, val_trees), (test_file, test_trees)]:
        with open(path, 'w') as f:
            f.write('\n'.join(t.strip() for t in trees) + '\n')
    print(f'  train={len(train_trees)}, val={len(val_trees)}, test(dummy)={len(test_trees)}')

    # Run preprocess.py
    out_prefix = os.path.join(cond_dir, cond)
    cmd = [
        'python', f'{RNNG_DIR}/preprocess.py',
        '--trainfile',    train_file,
        '--valfile',      val_file,
        '--testfile',     test_file,
        '--outputfile',   out_prefix,
        '--vocabminfreq', '1',
        '--unkmethod',    'berkeleyrule',
    ]
    print(f'  Running preprocess.py...')
    subprocess.run(cmd, check=True, cwd=RNNG_DIR)
    print(f'[{cond}] Preprocessing done.')

## Step 7 — Train RNNG (5 conditions × 3 seeds)

Each run saves a checkpoint to Drive.  If the session disconnects, re-run from this cell
— already-finished checkpoints are skipped automatically.

Rough T4 estimates: ~20–40 min per condition per seed at 150k sentences.  Full 3-seed
sweep ≈ 5–10 hours.

In [ ]:
# 7 — Train RNNG for all conditions × seeds
for seed in SEEDS:
    for cond in CONDITIONS:
        ckpt_path  = os.path.join(DRIVE_DIR, 'checkpoints', f'rnng_{cond}_seed{seed}.pt')
        cond_dir   = os.path.join(PARSE_DIR, cond)
        train_json = os.path.join(cond_dir, f'{cond}-train.json')
        val_json   = os.path.join(cond_dir, f'{cond}-val.json')

        if os.path.exists(ckpt_path):
            print(f'[{cond} seed{seed}] Checkpoint exists — skipping.')
            continue

        print(f'\n====== Training {cond} seed{seed} ======')
        cmd = [
            'python', f'{RNNG_DIR}/train.py',
            '--train_file',       train_json,
            '--val_file',         val_json,
            '--save_path',        ckpt_path,
            '--fixed_stack',
            '--strategy',         'top_down',
            '--w_dim',            '256',
            '--h_dim',            '256',
            '--num_layers',       '2',
            '--dropout',          '0.3',
            '--optimizer',        'adam',
            '--lr',               '0.001',
            '--batch_size',       '128',
            '--batch_token_size', '15000',
            '--num_epochs',       str(EPOCHS),
            '--gpu',              '0',
            '--seed',             str(seed),
        ]
        subprocess.run(cmd, check=True, cwd=RNNG_DIR)
        print(f'[{cond} seed{seed}] Done — saved to {ckpt_path}')

## Step 8 — Eval: minimal pairs → beam_search.py → metrics

For each pair `(prefix, correct_verb, incorrect_verb)` we write two raw-text files:

```
correct.txt:    prefix tokens + correct_verb   (one sentence per line)
incorrect.txt:  prefix tokens + incorrect_verb
```

`beam_search.py` runs word-synchronous beam search on raw text and outputs per-word
surprisals (`-log P(word | context)` in nats).  We extract the surprisal at the last
token position (the verb) and compare: if `surp(correct) < surp(incorrect)` the model
chose correctly.

Surprisal files are cached to Drive — re-run this cell to resume after a disconnect.

In [ ]:
# 8a — Build the held-out minimal-pair test set (same 20k rows as LSTM eval)
import math
from data_utils import get_test_indices, ORIGINAL_FILE
from minimal_pairs import build_minimal_pairs

test_idx  = get_test_indices()
orig_path = os.path.join(DATA_DIR, ORIGINAL_FILE)
# vocab=None: do NOT filter by LSTM vocab; RNNG has its own vocab
all_pairs, skipped = build_minimal_pairs(orig_path, test_idx, vocab=None)
print(f'Minimal pairs: {len(all_pairs)}, skipped: {skipped}')

In [ ]:
# 8b — Run beam_search.py for each condition × seed and compute metrics

def parse_surprisal_file(surp_file, n_sents):
    """Parse beam_search.py lm_output_file.
    Returns list of lists: sent_surps[i][j] = surprisal of word j in sentence i (nats).
    Format per line: sent_i TAB token_i TAB orig_token TAB model_token TAB word_surp TAB pieces
    """
    sent_surps = [[] for _ in range(n_sents)]
    with open(surp_file) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('---') or line.startswith('perplexity'):
                continue
            parts = line.split('\t')
            if len(parts) < 5:
                continue
            try:
                si, ti, surp = int(parts[0]), int(parts[1]), float(parts[4])
            except ValueError:
                continue
            if si < n_sents:
                while len(sent_surps[si]) <= ti:
                    sent_surps[si].append(None)
                sent_surps[si][ti] = surp
    return sent_surps


def summarize_rnng(pairs, surp_correct, surp_incorrect):
    """Compute the same metrics as the LSTM evaluate.py."""
    def acc(mask):
        idx = [k for k, m in enumerate(mask) if m]
        if not idx:
            return float('nan'), 0
        n_correct = sum(1 for k in idx if surp_correct[k] < surp_incorrect[k])
        return n_correct / len(idx), len(idx)

    attr    = [p['has_attractor'] for p in pairs]
    noattr  = [not a for a in attr]
    acc_all,    n_all    = acc([True] * len(pairs))
    acc_attr,   n_attr   = acc(attr)
    acc_noattr, n_noattr = acc(noattr)
    # surp_correct is in nats; convert to bits for comparability with LSTM metric
    mean_surp = sum(surp_correct) / max(len(surp_correct), 1) / math.log(2)
    gap = (acc_noattr - acc_attr) if (n_attr and n_noattr) else float('nan')
    return {
        'acc_all':                    acc_all,
        'n_all':                      n_all,
        'acc_no_attractor':           acc_noattr,
        'n_no_attractor':             n_noattr,
        'acc_attractor':              acc_attr,
        'n_attractor':                n_attr,
        'attractor_gap':              gap,
        'mean_surprisal_correct_bits': mean_surp,
    }


RESULTS_CSV = os.path.join(REPO_DIR, 'results', 'rnng_eval_results.csv')
os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
COLS = ['model', 'condition', 'rate', 'seed',
        'acc_all', 'acc_no_attractor', 'acc_attractor', 'attractor_gap',
        'mean_surprisal_correct_bits',
        'n_all', 'n_attractor', 'n_no_attractor', 'checkpoint']
write_header = not os.path.exists(RESULTS_CSV)

for seed in SEEDS:
    for cond in CONDITIONS:
        ckpt_path = os.path.join(DRIVE_DIR, 'checkpoints', f'rnng_{cond}_seed{seed}.pt')
        if not os.path.exists(ckpt_path):
            print(f'[{cond} seed{seed}] No checkpoint — skipping eval.')
            continue

        eval_dir = os.path.join(DRIVE_DIR, 'eval_tmp', f'{cond}_seed{seed}')
        os.makedirs(eval_dir, exist_ok=True)

        correct_txt   = os.path.join(eval_dir, 'correct.txt')
        incorrect_txt = os.path.join(eval_dir, 'incorrect.txt')

        # Write raw-text eval files (prefix + verb, one sentence per line)
        if not os.path.exists(correct_txt):
            with open(correct_txt, 'w') as fc, open(incorrect_txt, 'w') as fi:
                for p in all_pairs:
                    fc.write(' '.join(p['prefix'] + [p['correct']])   + '\n')
                    fi.write(' '.join(p['prefix'] + [p['incorrect']]) + '\n')

        # Run beam_search.py for correct and incorrect files
        for txt_file, surp_file in [
            (correct_txt,   os.path.join(eval_dir, 'surp_correct.txt')),
            (incorrect_txt, os.path.join(eval_dir, 'surp_incorrect.txt')),
        ]:
            if os.path.exists(surp_file):
                print(f'  [{cond} seed{seed}] {os.path.basename(surp_file)} cached — skipping.')
                continue
            print(f'  [{cond} seed{seed}] Running beam_search on {os.path.basename(txt_file)}...')
            cmd = [
                'python', f'{RNNG_DIR}/beam_search.py',
                '--test_file',        txt_file,
                '--model_file',       ckpt_path,
                '--lm_output_file',   surp_file,
                '--beam_size',        str(EVAL_BEAM),
                '--word_beam_size',   str(EVAL_WORD_BEAM),
                '--shift_size',       str(EVAL_SHIFT),
                '--batch_size',       str(EVAL_BATCH),
                '--stack_size_bound', str(EVAL_STACK),
                '--gpu',              '0',
                '--device',           'cuda',
            ]
            subprocess.run(cmd, check=True, cwd=RNNG_DIR)

        # Parse surprisal files
        surp_correct_file   = os.path.join(eval_dir, 'surp_correct.txt')
        surp_incorrect_file = os.path.join(eval_dir, 'surp_incorrect.txt')
        cs_per_sent = parse_surprisal_file(surp_correct_file,   len(all_pairs))
        is_per_sent = parse_surprisal_file(surp_incorrect_file, len(all_pairs))

        # Extract verb surprisal (= last token = position len(prefix))
        valid_pairs, surp_c, surp_i = [], [], []
        for k, p in enumerate(all_pairs):
            verb_pos = len(p['prefix'])
            cs = cs_per_sent[k]
            is_ = is_per_sent[k]
            if (cs and verb_pos < len(cs) and cs[verb_pos] is not None and
                    is_ and verb_pos < len(is_) and is_[verb_pos] is not None):
                valid_pairs.append(p)
                surp_c.append(cs[verb_pos])
                surp_i.append(is_[verb_pos])

        print(f'[{cond} seed{seed}] Valid pairs: {len(valid_pairs)}/{len(all_pairs)}')

        res = summarize_rnng(valid_pairs, surp_c, surp_i)
        res.update({
            'model':      'rnng',
            'condition':  cond,
            'rate':       RATES[cond],
            'seed':       seed,
            'checkpoint': os.path.basename(ckpt_path),
        })
        print(f'  acc_all={res["acc_all"]:.3f}  attractor_gap={res["attractor_gap"]:.3f}')

        with open(RESULTS_CSV, 'a', newline='') as f:
            import csv as csv_module
            w = csv_module.DictWriter(f, fieldnames=COLS)
            if write_header:
                w.writeheader()
                write_header = False
            w.writerow({k: res.get(k) for k in COLS})

print('\nEval complete.')

## Step 9 — Results summary + comparison plots

In [ ]:
# 9a — Copy results CSV to Drive as backup
import shutil
drive_csv = os.path.join(DRIVE_DIR, 'results', 'rnng_eval_results.csv')
shutil.copy(RESULTS_CSV, drive_csv)
print(f'Backed up to {drive_csv}')

# 9b — Display results table
import pandas as pd
df = pd.read_csv(RESULTS_CSV)
print(df[['model', 'condition', 'rate', 'seed', 'acc_all', 'attractor_gap']].to_string(index=False))

In [ ]:
# 9c — Generate LSTM vs RNNG comparison plots
lstm_csv = os.path.join(REPO_DIR, 'results', 'eval_results.csv')
rnng_csv = RESULTS_CSV

result = subprocess.run(
    ['python', 'src/plots_compare.py',
     '--lstm_csv', lstm_csv,
     '--rnng_csv', rnng_csv,
     '--out_dir',  'results'],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[:500])

from IPython.display import Image, display
for fig in ['compare_acc_vs_noise.png', 'compare_attractor_gap.png', 'compare_acc_by_attractor.png']:
    p = os.path.join(REPO_DIR, 'results', fig)
    if os.path.exists(p):
        display(Image(p))

## Step 10 — Commit results and push to repo

Run the cell below, then download `rnng_eval_results.csv` and the comparison PNGs from
the Drive folder, commit them locally, and push — **or** run the git commands directly
from Colab if the repo remote is authenticated.

Files to commit:
- `results/rnng_eval_results.csv`
- `results/compare_acc_vs_noise.png`
- `results/compare_attractor_gap.png`
- `results/compare_acc_by_attractor.png`
- `notebooks/rnng_colab.ipynb` (this notebook with outputs)

In [ ]:
# 10 — Optional: push directly from Colab (requires GitHub credentials)
# Uncomment and fill in your token if you want to push from here.

# GITHUB_TOKEN = 'ghp_...'   # Personal access token with repo scope
# GITHUB_EMAIL = 'j2wilke@ucsd.edu'
# GITHUB_NAME  = 'Jackson Wilke'

# !cd {REPO_DIR} && git config user.email "{GITHUB_EMAIL}"
# !cd {REPO_DIR} && git config user.name  "{GITHUB_NAME}"
# !cd {REPO_DIR} && git remote set-url origin https://{GITHUB_TOKEN}@github.com/blue-octopus235/FinalProject-LMs-CogModels-DSC291.git
# !cd {REPO_DIR} && git add results/rnng_eval_results.csv results/compare_*.png notebooks/rnng_colab.ipynb
# !cd {REPO_DIR} && git commit -m "Add RNNG eval results and comparison plots"
# !cd {REPO_DIR} && git push

# Otherwise, list the files to copy from Drive:
print('Files to commit:')
for f in ['results/rnng_eval_results.csv',
          'results/compare_acc_vs_noise.png',
          'results/compare_attractor_gap.png',
          'results/compare_acc_by_attractor.png']:
    p = os.path.join(REPO_DIR, f)
    print(f'  {f}: {"EXISTS" if os.path.exists(p) else "MISSING"}')